#  Forward pass

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann 
* https://pytorch.org/docs/stable/generated/torch.matmul.html
* https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/
* https://machinelearningmastery.com/loss-and-loss-functions-for-training-deep-learning-neural-networks/

In [1]:
# Install required packages
try:
    import torch
    print("PyTorch already installed")
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--trusted-host", "pypi.org", "--trusted-host", "pypi.python.org", "--trusted-host", "files.pythonhosted.org"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "matplotlib", "--trusted-host", "pypi.org", "--trusted-host", "pypi.python.org", "--trusted-host", "files.pythonhosted.org"])
    print("Packages installed successfully")

PyTorch already installed


In [2]:
# Import libraries
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
print("Libraries imported successfully")

Libraries imported successfully


## Задачи для совместного разбора

### 1. Реализация нейрона
Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте нейрон с заданными весами `weights` и `bias`. Пропустите вектор `inputs` через нейрон и выведите результат.

In [3]:
class Neuron:
    def __init__(self, weights, bias):
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.bias = torch.tensor(bias, dtype=torch.float32)
    
    def forward(self, inputs):
        inputs = torch.tensor(inputs, dtype=torch.float32)
        return torch.matmul(inputs, self.weights) + self.bias

# Test data
inputs = torch.tensor([1.0, 2.0, 3.0, 4.0])
weights = torch.tensor([-0.2, 0.3, -0.5, 0.7])
bias = 3.14

# Test neuron
neuron = Neuron(weights, bias)
result = neuron.forward(inputs)
print(f"Input: {inputs}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"Output: {result}")
print(f"Manual calculation: {torch.sum(inputs * weights) + bias}")

Input: tensor([1., 2., 3., 4.])
Weights: tensor([-0.2000,  0.3000, -0.5000,  0.7000])
Bias: 3.14
Output: 4.840000152587891
Manual calculation: 4.840000152587891


/var/folders/33/_3k27c410dg26xf71rr98ll40000gn/T/ipykernel_35884/1615340220.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.weights = torch.tensor(weights, dtype=torch.float32)
/var/folders/33/_3k27c410dg26xf71rr98ll40000gn/T/ipykernel_35884/1615340220.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs = torch.tensor(inputs, dtype=torch.float32)


### 2. Реализация функции активации ReLU
Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ReLU.

In [4]:
class ReLU:
    def forward(self, inputs):
        return torch.maximum(torch.zeros_like(inputs), inputs)

# Test ReLU
relu = ReLU()
test_matrix = torch.randn(4, 3)
print(f"Input matrix:\n{test_matrix}")
print(f"\nReLU output:\n{relu.forward(test_matrix)}")

# Verification
relu_output = relu.forward(test_matrix)
print(f"\nVerification - negative values become zero:")
print(f"Original < 0: {test_matrix < 0}")
print(f"ReLU output == 0 where original < 0: {torch.all((relu_output == 0) == (test_matrix < 0))}")

Input matrix:
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094]])

ReLU output:
tensor([[0.3367, 0.1288, 0.2345],
        [0.2303, 0.0000, 0.0000],
        [2.2082, 0.0000, 0.4617],
        [0.2674, 0.5349, 0.8094]])

Verification - negative values become zero:
Original < 0: tensor([[False, False, False],
        [False,  True,  True],
        [False,  True, False],
        [False, False, False]])
ReLU output == 0 where original < 0: True


### 3. Реализация функции потерь MSE
Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь MSE.

In [5]:
class MSELoss:
    def forward(self, y_pred, y_true):
        diff = y_pred - y_true
        squared_diff = diff ** 2
        return torch.mean(squared_diff)

# Test MSE
y_pred = torch.tensor([1.0, 3.0, 5.0])
y_true = torch.tensor([2.0, 3.0, 4.0])

mse_loss = MSELoss()
loss_value = mse_loss.forward(y_pred, y_true)
print(f"Predicted: {y_pred}")
print(f"True: {y_true}")
print(f"MSE Loss: {loss_value}")
print(f"Manual calculation: {torch.mean((y_pred - y_true)**2)}")

Predicted: tensor([1., 3., 5.])
True: tensor([2., 3., 4.])
MSE Loss: 0.6666666865348816
Manual calculation: 0.6666666865348816


## Задачи для самостоятельного решения

### Создание полносвязных слоев

**1.1** Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте полносвязный слой из `n_neurons` нейронов с `n_features` весами у каждого нейрона (инициализируются из стандартного нормального распределения) и опциональным вектором смещения.

$$y = xW^T + b$$

Пропустите вектор `inputs` через слой и выведите результат. Результатом прогона сквозь слой должна быть матрица размера `batch_size` x `n_neurons`.

In [7]:
class Linear:
    def __init__(self, n_neurons, n_features, bias: bool = False):
        self.weights = torch.randn(n_neurons, n_features)
        self.use_bias = bias
        if bias:
            self.bias = torch.randn(n_neurons)
        else:
            self.bias = None
    
    def forward(self, inputs):
        output = torch.matmul(inputs, self.weights.T)
        if self.use_bias:
            output = output + self.bias
        return output

# Test data
inputs = torch.tensor([[1, 2, 3, 2.5], 
                       [2, 5, -1, 2], 
                       [-1.5, 2.7, 3.3, -0.8]])

print(f"Input shape: {inputs.shape}")

# Create layer with 5 neurons, each with 4 weights
linear_layer = Linear(n_neurons=5, n_features=4, bias=True)
output = linear_layer.forward(inputs)
print(f"Linear layer output shape: {output.shape}")
print(f"Linear layer output:\n{output}")

# Check dimensions
print(f"\nWeights shape: {linear_layer.weights.shape}")
print(f"Bias shape: {linear_layer.bias.shape if linear_layer.bias is not None else 'No bias'}")

Input shape: torch.Size([3, 4])
Linear layer output shape: torch.Size([3, 5])
Linear layer output:
tensor([[-0.6111, -6.2046,  0.9110,  1.1808,  3.9410],
        [ 0.2640, -8.1096, -3.2872, -2.2875,  4.3058],
        [-0.3405, -9.8804,  7.6673, -5.8796,  4.4636]])

Weights shape: torch.Size([5, 4])
Bias shape: torch.Size([5])


**1.2** Используя решение предыдущей задачи, создайте 2 полносвязных слоя и пропустите тензор `inputs` последовательно через эти два слоя. Количество нейронов в первом слое выберите произвольно, количество нейронов во втором слое выберите так, чтобы результатом прогона являлась матрица `batch_size x 7`.

In [8]:
# Task 1.2: Two sequential linear layers
# First layer: 4 inputs -> 10 neurons
# Second layer: 10 inputs -> 7 neurons (final size batch_size x 7)

layer1 = Linear(n_neurons=10, n_features=4, bias=True)
layer2 = Linear(n_neurons=7, n_features=10, bias=True)

print(f"Input shape: {inputs.shape}")

# Forward through first layer
output1 = layer1.forward(inputs)
print(f"After layer 1 shape: {output1.shape}")

# Forward through second layer
output2 = layer2.forward(output1)
print(f"Final output shape: {output2.shape}")
print(f"Final output:\n{output2}")

# Verify that final shape matches requirement batch_size x 7
assert output2.shape == (3, 7), f"Expected shape (3, 7), got {output2.shape}"
print("✓ Shape verification passed!")

Input shape: torch.Size([3, 4])
After layer 1 shape: torch.Size([3, 10])
Final output shape: torch.Size([3, 7])
Final output:
tensor([[  5.6725,   3.8567,   1.6868,  14.3292, -14.7362,  -6.4734, -25.3238],
        [ -2.2912,  20.9368,  -4.1582,   9.7217,  -8.8855, -17.9996, -22.3851],
        [ 17.1226,  -1.5623,   3.6164,  29.5273, -11.3124,   8.3512,  -9.8668]])
✓ Shape verification passed!


### Создание функций активации

**2.1** Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации softmax.

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации. Строки матрицы трактовать как выходы линейного слоя некоторого классификатора для 4 различных примеров. Функция должна применяться к переданной на вход матрице построчно.

In [9]:
class Softmax:
    def forward(self, inputs):
        # For numerical stability, subtract max along each row
        inputs_shifted = inputs - torch.max(inputs, dim=1, keepdim=True)[0]
        exp_values = torch.exp(inputs_shifted)
        probabilities = exp_values / torch.sum(exp_values, dim=1, keepdim=True)
        return probabilities

# Test Softmax
softmax = Softmax()
test_matrix = torch.randn(4, 3)
print(f"Input matrix:\n{test_matrix}")

softmax_output = softmax.forward(test_matrix)
print(f"\nSoftmax output:\n{softmax_output}")

# Verify that row sums equal 1
row_sums = torch.sum(softmax_output, dim=1)
print(f"\nRow sums (should be ~1.0): {row_sums}")
print(f"All sums close to 1.0: {torch.allclose(row_sums, torch.ones_like(row_sums))}")

# Verify that all values are non-negative
print(f"All values non-negative: {torch.all(softmax_output >= 0)}")

Input matrix:
tensor([[-0.2732, -1.0541,  0.0887],
        [-0.2586, -0.4564, -1.1848],
        [-1.1518, -1.1096,  0.8142],
        [-0.1609, -1.2246,  0.4401]])

Softmax output:
tensor([[0.3455, 0.1583, 0.4962],
        [0.4511, 0.3702, 0.1787],
        [0.1089, 0.1136, 0.7776],
        [0.3155, 0.1089, 0.5755]])

Row sums (should be ~1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000])
All sums close to 1.0: True
All values non-negative: True


**2.2** Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ELU.

Создайте матрицу размера 4x3, заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

In [10]:
class ELU:
    def __init__(self, alpha: float):
        self.alpha = alpha
    
    def forward(self, inputs):
        return torch.where(inputs > 0, inputs, self.alpha * (torch.exp(inputs) - 1))

# Test ELU
elu = ELU(alpha=1.0)
test_matrix = torch.randn(4, 3)
print(f"Input matrix:\n{test_matrix}")

elu_output = elu.forward(test_matrix)
print(f"\nELU output:\n{elu_output}")

# Verify logic: positive values unchanged
positive_mask = test_matrix > 0
print(f"\nPositive values unchanged: {torch.allclose(test_matrix[positive_mask], elu_output[positive_mask])}")

# Verify that negative values are processed correctly
negative_mask = test_matrix <= 0
expected_negative = elu.alpha * (torch.exp(test_matrix[negative_mask]) - 1)
print(f"Negative values processed correctly: {torch.allclose(elu_output[negative_mask], expected_negative)}")

Input matrix:
tensor([[-1.4477, -0.5346,  0.4590],
        [ 1.0935, -0.2637, -0.2296],
        [ 2.2003, -0.7842, -2.3885],
        [ 1.1878, -0.0755,  0.8630]])

ELU output:
tensor([[-0.7649, -0.4141,  0.4590],
        [ 1.0935, -0.2318, -0.2051],
        [ 2.2003, -0.5435, -0.9082],
        [ 1.1878, -0.0727,  0.8630]])

Positive values unchanged: True
Negative values processed correctly: True


### Создание функций потерь

**3.1** Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь CrossEntropyLoss.

Создайте полносвязный слой с 3 нейронами и прогоните через него батч `inputs`. Полученный результат пропустите через функцию активации Softmax. Посчитайте значение функции потерь, трактуя вектор `y` как вектор правильных ответов.

In [11]:
class CrossEntropyLoss:
    def forward(self, y_pred, y_true):
        # y_pred - logits (before softmax), y_true - true class indices
        # Apply log_softmax for numerical stability
        log_softmax = torch.log_softmax(y_pred, dim=1)
        
        # Select log probabilities for correct classes
        nll_loss = -log_softmax[range(len(y_true)), y_true]
        
        # Return mean loss
        return torch.mean(nll_loss)

# Test data
inputs = torch.tensor([[1, 2, 3, 2.5], 
                       [2, 5, -1, 2], 
                       [-1.5, 2.7, 3.3, -0.8]])
y = torch.tensor([1, 0, 0])

# Test CrossEntropy with full pipeline
print(f"Input batch shape: {inputs.shape}")
print(f"True labels: {y}")

# Create linear layer with 3 neurons
linear_classifier = Linear(n_neurons=3, n_features=4, bias=True)

# Forward through layer
logits = linear_classifier.forward(inputs)
print(f"\nLogits from linear layer:\n{logits}")

# Apply Softmax
softmax = Softmax()
probabilities = softmax.forward(logits)
print(f"\nProbabilities after Softmax:\n{probabilities}")

# Calculate CrossEntropy loss
ce_loss = CrossEntropyLoss()
loss_value = ce_loss.forward(logits, y)
print(f"\nCrossEntropy Loss: {loss_value}")

# Verify with PyTorch built-in function
pytorch_loss = F.cross_entropy(logits, y)
print(f"PyTorch CrossEntropy (verification): {pytorch_loss}")
print(f"Results match: {torch.allclose(loss_value, pytorch_loss, atol=1e-6)}")

Input batch shape: torch.Size([3, 4])
True labels: tensor([1, 0, 0])

Logits from linear layer:
tensor([[ 6.1105,  3.0824, -8.1683],
        [13.2326,  1.9128, -5.7811],
        [ 2.4029,  1.5555, -5.0581]])

Probabilities after Softmax:
tensor([[9.5382e-01, 4.6177e-02, 6.0016e-07],
        [9.9999e-01, 1.2129e-05, 5.5264e-09],
        [6.9974e-01, 2.9986e-01, 4.0242e-04]])

CrossEntropy Loss: 1.1441131830215454
PyTorch CrossEntropy (verification): 1.1441131830215454
Results match: True


**3.2** Модифицируйте MSE, добавив L2-регуляризацию.

$$MSE_R = MSE + \lambda\sum_{i=1}^{m}w_i^2$$

где $\lambda$ - коэффициент регуляризации; $w_i$ - веса модели.

In [12]:
class MSERegularized:
    def __init__(self, lambda_):
        self.lambda_ = lambda_
    
    def data_loss(self, y_pred, y_true):
        # Calculate first term from formula - regular MSE
        diff = y_pred - y_true
        return torch.mean(diff ** 2)
    
    def reg_loss(self, weights):
        # Calculate second term from formula - L2 regularization
        return self.lambda_ * torch.sum(weights ** 2)
    
    def forward(self, y_pred, y_true, weights):
        return self.data_loss(y_pred, y_true) + self.reg_loss(weights)

# Test data
y_pred = torch.tensor([-0.5, 1, 1.7])
y_true = torch.tensor([0, 0.6, 2.3])
weights = torch.normal(0, 5, (10, 1))

# Test MSE with L2 regularization
lambda_reg = 0.01
mse_reg = MSERegularized(lambda_reg)

print(f"Predicted: {y_pred}")
print(f"True: {y_true}")
print(f"Weights shape: {weights.shape}")
print(f"Lambda: {lambda_reg}")

# Calculate loss components separately
data_loss = mse_reg.data_loss(y_pred, y_true)
reg_loss = mse_reg.reg_loss(weights)
total_loss = mse_reg.forward(y_pred, y_true, weights)

print(f"\nData loss (MSE): {data_loss}")
print(f"Regularization loss: {reg_loss}")
print(f"Total loss: {total_loss}")
print(f"Manual verification: {data_loss + reg_loss}")

# Verify that total loss is higher than regular MSE loss
regular_mse = MSELoss()
regular_loss = regular_mse.forward(y_pred, y_true)
print(f"\nRegular MSE loss: {regular_loss}")
print(f"Regularized loss is higher: {total_loss > regular_loss}")

Predicted: tensor([-0.5000,  1.0000,  1.7000])
True: tensor([0.0000, 0.6000, 2.3000])
Weights shape: torch.Size([10, 1])
Lambda: 0.01

Data loss (MSE): 0.2566666305065155
Regularization loss: 1.9928687810897827
Total loss: 2.249535322189331
Manual verification: 2.249535322189331

Regular MSE loss: 0.2566666305065155
Regularized loss is higher: True
